# Mô hình 2 - XGBoost Direct Multi-Horizon

XGBoost được triển khai theo chiến lược **Direct Forecast**: 24 bộ hồi quy độc lập
dự đoán lần lượt PM2.5 tại `t+1,...,t+24`. Cách này tránh truyền sai số từ bước trước
sang bước sau như dự báo đệ quy.

Bộ siêu tham số regularization kế thừa kết quả tuning của thí nghiệm `t+24` để giới
hạn chi phí từ 24 lần huấn luyện. Mỗi horizon vẫn có early stopping riêng trên
Validation; Test hoàn toàn không tham gia chọn số boosting round.


In [1]:
from pathlib import Path
import json
import random
from typing import Any

import joblib
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "mathtext.fontset": "dejavusans",
    "axes.unicode_minus": False,
})
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Nhận diện project bằng cấu trúc, không phụ thuộc tên thư mục tạm của project.
CURRENT_DIR = Path.cwd().resolve()
SEARCH_DIRS = [CURRENT_DIR, CURRENT_DIR.parent]
SEARCH_DIRS.extend(path for path in CURRENT_DIR.iterdir() if path.is_dir())
PROJECT_CANDIDATES = []
for candidate in SEARCH_DIRS:
    data_file = candidate / "data" / "processed" / "pm25_training_data_enriched.csv"
    notebook_marker = candidate / "model" / "0_multihorizon_data_preparation.ipynb"
    if notebook_marker.exists() and data_file.exists():
        resolved = candidate.resolve()
        if resolved not in PROJECT_CANDIDATES:
            PROJECT_CANDIDATES.append(resolved)

if len(PROJECT_CANDIDATES) == 1:
    PROJECT_ROOT = PROJECT_CANDIDATES[0]
elif not PROJECT_CANDIDATES:
    raise FileNotFoundError(
        "Không tìm thấy project chứa đồng thời model và "
        "data/processed/pm25_training_data_enriched.csv."
    )
else:
    raise RuntimeError(
        "Có nhiều project phù hợp; hãy mở Jupyter tại đúng thư mục gốc cần chạy: "
        + ", ".join(str(path) for path in PROJECT_CANDIDATES)
    )

MODEL_DIR = PROJECT_ROOT / "model"
RESULTS_DIR = MODEL_DIR / "results"
CANDIDATES_DIR = MODEL_DIR / "candidates"
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "pm25_training_data_enriched.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CANDIDATES_DIR.mkdir(parents=True, exist_ok=True)

HORIZONS = np.arange(1, 25, dtype=int)
TARGET_COLUMNS = [f"target_pm25_t_plus_{h}" for h in HORIZONS]
MAX_HORIZON = int(HORIZONS.max())

POLLUTANT_FEATURES = ["pm25", "pm10", "o3", "no2", "so2", "co"]
WEATHER_FEATURES = [
    "temp", "humidity", "wind_speed", "wind_dir", "precip", "pressure", "cloud_cover"
]
TEMPORAL_FEATURES = ["hour", "day_of_week", "month", "is_weekend", "day_of_year"]
HISTORY_FEATURES = [
    "pm25_lag_1h", "pm25_lag_3h", "pm25_lag_6h", "pm25_lag_12h",
    "pm25_lag_24h", "pm25_lag_48h", "pm25_lag_72h", "pm25_lag_96h",
    "pm25_lag_120h", "pm25_lag_144h", "pm25_lag_168h",
    "pm25_roll_6h", "pm25_roll_12h", "pm25_roll_24h", "pm25_roll_72h",
    "pm25_roll_168h", "pm25_std_6h", "pm25_std_12h", "pm25_std_24h",
    "pm25_std_72h", "pm25_std_168h", "pm25_min_24h", "pm25_max_24h",
    "pm25_delta_1h", "pm25_delta_3h", "pm25_delta_24h",
    "pm25_roll_ratio_6h_24h", "pm25_roll_ratio_24h_72h",
    "pm25_same_hour_mean_7d", "pm25_same_hour_median_7d",
    "pm25_same_hour_std_7d", "pm25_same_hour_min_7d",
    "pm25_same_hour_max_7d", "pm25_same_hour_ratio_7d", "pm25_weekly_delta",
]
ENGINEERED_FEATURES = [
    "ventilation_index", "humid_stagnation", "rain_flag", "calm_wind",
    "high_humidity", "wind_x", "wind_y", "hour_sin", "hour_cos",
    "day_sin", "day_cos", "month_sin", "month_cos", "pm25_pm10_ratio",
    "no2_co_ratio",
]
NUMERIC_FEATURES = (
    POLLUTANT_FEATURES + WEATHER_FEATURES + TEMPORAL_FEATURES
    + HISTORY_FEATURES + ENGINEERED_FEATURES
)
CATEGORICAL_FEATURES = ["city", "season"]

print(f"Project root: {PROJECT_ROOT}")
print(f"Data: {DATA_PATH}")


Project root: D:\Project123456\aqi-vietnam\aqi-vietnam4
Data: D:\Project123456\aqi-vietnam\aqi-vietnam4\data\processed\pm25_training_data_enriched.csv


In [2]:
def make_multihorizon_frame(data: pd.DataFrame) -> pd.DataFrame:
    """Ghép chính xác PM2.5 tại t+1,...,t+24 theo city và timestamp."""
    frame = data.copy()
    frame["datetime"] = pd.to_datetime(frame["datetime"])
    frame = frame.sort_values(["city", "datetime"]).reset_index(drop=True)
    if frame.duplicated(["city", "datetime"]).any():
        raise ValueError("Dữ liệu có city/datetime trùng, không thể ghép target chính xác.")

    lookup = frame.set_index(["city", "datetime"])["pm25"]
    for horizon, column in zip(HORIZONS, TARGET_COLUMNS):
        keys = pd.MultiIndex.from_arrays(
            [frame["city"], frame["datetime"] + pd.to_timedelta(horizon, unit="h")],
            names=["city", "datetime"],
        )
        frame[column] = lookup.reindex(keys).to_numpy(dtype=float)
    frame["forecast_end"] = frame["datetime"] + pd.Timedelta(hours=MAX_HORIZON)
    return frame


def make_temporal_split(
    frame: pd.DataFrame,
    train_fraction: float = 0.70,
    validation_fraction: float = 0.15,
    purge_hours: int = MAX_HORIZON,
) -> dict[str, Any]:
    """Chia theo forecast_end để không có cửa sổ target giao nhau giữa các tập."""
    unique_ends = pd.Series(frame["forecast_end"].dropna().sort_values().unique())
    train_cut = pd.Timestamp(unique_ends.iloc[int(len(unique_ends) * train_fraction)])
    val_cut = pd.Timestamp(
        unique_ends.iloc[int(len(unique_ends) * (train_fraction + validation_fraction))]
    )
    purge = pd.Timedelta(hours=purge_hours)
    forecast_end = pd.to_datetime(frame["forecast_end"])
    masks = {
        "train": forecast_end < train_cut,
        "validation": (forecast_end >= train_cut + purge) & (forecast_end < val_cut),
        "test": forecast_end >= val_cut + purge,
    }
    if any(not mask.any() for mask in masks.values()):
        raise ValueError("Temporal split tạo ra ít nhất một tập rỗng.")
    return {
        "masks": masks,
        "train_cut": train_cut,
        "val_cut": val_cut,
        "purge_hours": purge_hours,
    }


def split_summary(frame: pd.DataFrame, split: dict[str, Any]) -> dict[str, Any]:
    summary: dict[str, Any] = {
        "strategy": "global chronological 70/15/15 by forecast_end with 24-hour purge gaps",
        "train_cut": split["train_cut"].isoformat(),
        "validation_cut": split["val_cut"].isoformat(),
        "purge_hours": int(split["purge_hours"]),
        "horizons": HORIZONS.tolist(),
    }
    for name, mask in split["masks"].items():
        part = frame.loc[mask]
        summary[name] = {
            "rows": int(len(part)),
            "source_start": part["datetime"].min().isoformat(),
            "source_end": part["datetime"].max().isoformat(),
            "forecast_end_start": part["forecast_end"].min().isoformat(),
            "forecast_end_end": part["forecast_end"].max().isoformat(),
        }
    return summary


def load_model_frame() -> tuple[pd.DataFrame, dict[str, Any], pd.DataFrame]:
    raw = pd.read_csv(DATA_PATH, low_memory=False)
    raw["datetime"] = pd.to_datetime(raw["datetime"])
    supervised = make_multihorizon_frame(raw)
    required = NUMERIC_FEATURES + CATEGORICAL_FEATURES + TARGET_COLUMNS
    missing_columns = sorted(set(required) - set(supervised.columns))
    if missing_columns:
        raise KeyError(f"Thiếu cột cần thiết: {missing_columns}")
    frame = supervised.dropna(subset=required).copy().reset_index(drop=True)
    split = make_temporal_split(frame)
    manifest = split_summary(frame, split)
    (RESULTS_DIR / "multihorizon_temporal_split.json").write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return frame, split, raw


def build_feature_matrix(
    frame: pd.DataFrame,
    feature_columns: list[str] | None = None,
) -> pd.DataFrame:
    matrix = pd.get_dummies(
        frame[NUMERIC_FEATURES + CATEGORICAL_FEATURES],
        columns=CATEGORICAL_FEATURES,
        drop_first=False,
        dtype=float,
    )
    if feature_columns is None:
        return matrix.astype(np.float32)
    for column in feature_columns:
        if column not in matrix:
            matrix[column] = 0.0
    return matrix.reindex(columns=feature_columns, fill_value=0.0).astype(np.float32)


def regression_metrics(actual: Any, predicted: Any) -> dict[str, float]:
    y_true = np.asarray(actual, dtype=float)
    y_pred = np.clip(np.asarray(predicted, dtype=float), 0.0, None)
    return {
        "rmse_ug_m3": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae_ug_m3": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
        "bias_ug_m3": float(np.mean(y_pred - y_true)),
    }


def prediction_frame(
    frame: pd.DataFrame,
    mask: pd.Series,
    predictions: np.ndarray,
    model_name: str,
    split_name: str,
) -> pd.DataFrame:
    part = frame.loc[mask].reset_index(drop=True)
    actual = part[TARGET_COLUMNS].to_numpy(dtype=float)
    predicted = np.clip(np.asarray(predictions, dtype=float), 0.0, None)
    if predicted.shape != actual.shape:
        raise ValueError(f"Prediction shape {predicted.shape} khác target shape {actual.shape}.")
    rows = len(part)
    output = pd.DataFrame({
        "model": model_name,
        "split": split_name,
        "city": np.repeat(part["city"].to_numpy(), len(HORIZONS)),
        "source_time": np.repeat(part["datetime"].to_numpy(), len(HORIZONS)),
        "horizon": np.tile(HORIZONS, rows),
        "actual_pm25": actual.reshape(-1),
        "predicted_pm25": predicted.reshape(-1),
    })
    output["target_time"] = pd.to_datetime(output["source_time"]) + pd.to_timedelta(
        output["horizon"], unit="h"
    )
    output["abs_error_ug_m3"] = np.abs(output["actual_pm25"] - output["predicted_pm25"])
    return output


def metric_tables(predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    horizon_rows = []
    for (model, split_name, horizon), part in predictions.groupby(
        ["model", "split", "horizon"], sort=True
    ):
        horizon_rows.append({
            "model": model,
            "split": split_name,
            "horizon": int(horizon),
            "rows": int(len(part)),
            **regression_metrics(part["actual_pm25"], part["predicted_pm25"]),
        })
    by_horizon = pd.DataFrame(horizon_rows)

    city_rows = []
    for (model, split_name, city), part in predictions.groupby(
        ["model", "split", "city"], sort=True
    ):
        city_rows.append({
            "model": model,
            "split": split_name,
            "city": city,
            "rows": int(len(part)),
            **regression_metrics(part["actual_pm25"], part["predicted_pm25"]),
        })
    by_city = pd.DataFrame(city_rows)

    summary_rows = []
    for (model, split_name), part in predictions.groupby(["model", "split"], sort=True):
        horizon_part = by_horizon.loc[
            by_horizon["model"].eq(model) & by_horizon["split"].eq(split_name)
        ]
        summary_rows.append({
            "model": model,
            "split": split_name,
            "rows": int(len(part)),
            "mean_horizon_rmse_ug_m3": float(horizon_part["rmse_ug_m3"].mean()),
            "mean_horizon_mae_ug_m3": float(horizon_part["mae_ug_m3"].mean()),
            **{f"global_{key}": value for key, value in regression_metrics(
                part["actual_pm25"], part["predicted_pm25"]
            ).items()},
        })
    return by_horizon, by_city, pd.DataFrame(summary_rows)


def save_evaluation(model_slug: str, predictions: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    by_horizon, by_city, summary = metric_tables(predictions)
    predictions.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )
    by_horizon.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_by_horizon.csv",
        index=False,
        encoding="utf-8-sig",
    )
    by_city.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_by_city.csv",
        index=False,
        encoding="utf-8-sig",
    )
    summary.to_csv(
        RESULTS_DIR / f"{model_slug}_multihorizon_summary.csv",
        index=False,
        encoding="utf-8-sig",
    )
    return by_horizon, by_city, summary


In [3]:
import xgboost as xgb

MODEL_NAME = "XGBoost"
MODEL_SLUG = "xgboost"
XGB_PARAMS = {
    "n_estimators": 800,
    "learning_rate": 0.04,
    "max_depth": 3,
    "min_child_weight": 8,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.05,
    "reg_lambda": 3.0,
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "tree_method": "hist",
    "random_state": SEED,
    "n_jobs": -1,
    "early_stopping_rounds": 45,
}


In [4]:
model_frame, split, raw_data = load_model_frame()
matrix = build_feature_matrix(model_frame)
feature_columns = matrix.columns.tolist()
targets = model_frame[TARGET_COLUMNS].to_numpy(dtype=np.float32)

train_mask = split["masks"]["train"].to_numpy()
validation_mask = split["masks"]["validation"].to_numpy()
test_mask = split["masks"]["test"].to_numpy()

x_train, y_train = matrix.loc[train_mask], targets[train_mask]
x_validation, y_validation = matrix.loc[validation_mask], targets[validation_mask]
x_test, y_test = matrix.loc[test_mask], targets[test_mask]
print(x_train.shape, x_validation.shape, x_test.shape)


(68829, 75) (14679, 75) (14679, 75)


In [5]:
models: dict[int, xgb.XGBRegressor] = {}
histories = []
validation_predictions = np.empty_like(y_validation, dtype=float)
test_predictions = np.empty_like(y_test, dtype=float)

for output_index, horizon in enumerate(HORIZONS):
    print(f"[XGBoost {horizon:02d}/24] Đang huấn luyện target t+{horizon}")
    model = xgb.XGBRegressor(**XGB_PARAMS)
    model.fit(
        x_train,
        y_train[:, output_index],
        eval_set=[
            (x_train, y_train[:, output_index]),
            (x_validation, y_validation[:, output_index]),
        ],
        verbose=False,
    )
    models[int(horizon)] = model
    validation_predictions[:, output_index] = np.clip(model.predict(x_validation), 0.0, None)
    test_predictions[:, output_index] = np.clip(model.predict(x_test), 0.0, None)

    history = model.evals_result()
    rounds = len(history["validation_0"]["rmse"])
    histories.append(pd.DataFrame({
        "horizon": int(horizon),
        "round": np.arange(1, rounds + 1),
        "train_rmse": history["validation_0"]["rmse"],
        "validation_rmse": history["validation_1"]["rmse"],
    }))
    print(
        f"  best_iteration={model.best_iteration}, "
        f"best_validation_rmse={model.best_score:.3f}"
    )

learning_curves = pd.concat(histories, ignore_index=True)


[XGBoost 01/24] Đang huấn luyện target t+1


  best_iteration=798, best_validation_rmse=4.725
[XGBoost 02/24] Đang huấn luyện target t+2


  best_iteration=798, best_validation_rmse=7.527
[XGBoost 03/24] Đang huấn luyện target t+3


  best_iteration=534, best_validation_rmse=9.905
[XGBoost 04/24] Đang huấn luyện target t+4


  best_iteration=799, best_validation_rmse=11.779
[XGBoost 05/24] Đang huấn luyện target t+5


...

  best_iteration=199, best_validation_rmse=18.104
[XGBoost 21/24] Đang huấn luyện target t+21


  best_iteration=200, best_validation_rmse=18.193
[XGBoost 22/24] Đang huấn luyện target t+22


  best_iteration=200, best_validation_rmse=18.176
[XGBoost 23/24] Đang huấn luyện target t+23


  best_iteration=170, best_validation_rmse=18.165
[XGBoost 24/24] Đang huấn luyện target t+24


  best_iteration=220, best_validation_rmse=18.095


In [6]:
artifact = {
    "model": MODEL_NAME,
    "strategy": "24 independent direct XGBoost regressors",
    "models": models,
    "feature_columns": feature_columns,
    "horizons": HORIZONS.tolist(),
    "params": XGB_PARAMS,
}
joblib.dump(artifact, CANDIDATES_DIR / "xgboost_multihorizon.joblib", compress=3)
learning_curves.to_csv(
    RESULTS_DIR / "xgboost_multihorizon_learning_curves.csv",
    index=False,
    encoding="utf-8-sig",
)

predictions = pd.concat([
    prediction_frame(
        model_frame, split["masks"]["validation"], validation_predictions,
        MODEL_NAME, "validation"
    ),
    prediction_frame(
        model_frame, split["masks"]["test"], test_predictions,
        MODEL_NAME, "test"
    ),
], ignore_index=True)
by_horizon, by_city, summary = save_evaluation(MODEL_SLUG, predictions)
display(summary)
display(by_city)


,model,split,rows,mean_horizon_rmse_ug_m3,mean_horizon_mae_ug_m3,global_rmse_ug_m3,global_mae_ug_m3,global_r2,global_bias_ug_m3
0,XGBoost,test,352296,14.012522,8.910852,14.384410,8.910852,0.664944,-0.288406
1,XGBoost,validation,352296,15.851590,9.763921,16.270426,9.763921,0.581209,-1.824405


,model,split,city,rows,rmse_ug_m3,mae_ug_m3,r2,bias_ug_m3
0,XGBoost,test,Hà Nội,117432,21.874949,15.268627,0.523115,-2.487711
1,XGBoost,test,TP.HCM,117432,9.895209,6.932588,0.378830,0.539817
2,XGBoost,test,Đà Nẵng,117432,6.656218,4.531341,0.567271,1.082675
3,XGBoost,validation,Hà Nội,117432,23.039299,14.550077,0.400733,-4.561029
4,XGBoost,validation,TP.HCM,117432,13.580074,9.120260,0.433884,-1.854521
5,XGBoost,validation,Đà Nẵng,117432,8.885526,5.621424,0.548844,0.942335


In [7]:
test_curve = by_horizon.loc[by_horizon["split"].eq("test")]
axis = test_curve.plot(
    x="horizon", y=["rmse_ug_m3", "mae_ug_m3"], marker="o", figsize=(9, 4.5)
)
axis.set_xlabel("Chân trời dự báo (giờ)")
axis.set_ylabel("Sai số (µg/m³)")
axis.set_title("XGBoost: sai số Test theo chân trời")
axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()


C:\Users\nguyen\AppData\Local\Temp\ipykernel_2184\1551201773.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
selected_horizons = [1, 6, 12, 24]
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=False)
for axis, horizon in zip(axes.flat, selected_horizons):
    curve = learning_curves.loc[learning_curves["horizon"].eq(horizon)]
    axis.plot(curve["round"], curve["train_rmse"], label="Train RMSE")
    axis.plot(curve["round"], curve["validation_rmse"], label="Validation RMSE")
    axis.set_title(f"Target t+{horizon}")
    axis.set_xlabel("Boosting round")
    axis.set_ylabel("RMSE (µg/m³)")
    axis.grid(alpha=0.2)
axes[0, 0].legend()
plt.tight_layout()
plt.show()


C:\Users\nguyen\AppData\Local\Temp\ipykernel_2184\3720424303.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
val_summary = summary.loc[summary["split"].eq("validation")].iloc[0]
test_summary = summary.loc[summary["split"].eq("test")].iloc[0]
display(Markdown(
    f"**Nhận xét.** Validation RMSE trung bình theo horizon là "
    f"**{val_summary['mean_horizon_rmse_ug_m3']:.2f} µg/m³**; Test RMSE là "
    f"**{test_summary['mean_horizon_rmse_ug_m3']:.2f} µg/m³**. Bốn learning curve "
    "đại diện cho horizon ngắn, trung bình và dài, giúp kiểm tra khoảng cách "
    "Train–Validation và thời điểm early stopping."
))


**Nhận xét.** Validation RMSE trung bình theo horizon là **15.85 µg/m³**; Test RMSE là **14.01 µg/m³**. Bốn learning curve đại diện cho horizon ngắn, trung bình và dài, giúp kiểm tra khoảng cách Train–Validation và thời điểm early stopping.